# Phase 7 — Prior Case Retrieval (PCR)
## Experiment : Phase-1 Encoder + LoRA + Random Negatives




In [31]:
import os
# Set allocator option before substantial CUDA allocations.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import os
import pickle
import random
import time
from functools import partial
from dataclasses import dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint

print("PyTorch:", torch.__version__)


PyTorch: 2.11.0+cu130


In [32]:
# ------------------------------------------------------------
# GPU setup + aggressive cleanup for rerunning notebook cells
# ------------------------------------------------------------

def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        for gpu_id in range(torch.cuda.device_count()):
            with torch.cuda.device(gpu_id):
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
                torch.cuda.reset_peak_memory_stats()

clear_cuda_memory()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")


CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [33]:
# ------------------------------------------------------------
# Hugging Face login
# ------------------------------------------------------------

from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Hugging Face login successful.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [34]:
# ------------------------------------------------------------
# Download Phase-7 PCR files
# ------------------------------------------------------------

TRAIN_FILES = [
    "Phase_7_PCR/Train/train.json",
    "Phase_7_PCR/Train/train_candidates.json",
]
DEV_FILES = [
    "Phase_7_PCR/Dev/dev.json",
    "Phase_7_PCR/Dev/dev_candidates.json",
]
TEST_FILES = [
    "Phase_7_PCR/Test/test.json",
    "Phase_7_PCR/Test/test_candidates.json",
]

def download_files(files):
    paths = {}
    for filename in files:
        paths[filename] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        print(filename, "->", paths[filename])
    return paths

train_file_paths = download_files(TRAIN_FILES)
dev_file_paths = download_files(DEV_FILES)
test_file_paths = download_files(TEST_FILES)


Phase_7_PCR/Train/train.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Train/train.json
Phase_7_PCR/Train/train_candidates.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Train/train_candidates.json
Phase_7_PCR/Dev/dev.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Dev/dev.json
Phase_7_PCR/Dev/dev_candidates.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Dev/dev_candidates.json
Phase_7_PCR/Test/test.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Test/test.json
Phase_7_PCR/Test/test_candidates.json -> /root/.cache/hu

In [35]:
# ------------------------------------------------------------
# Load raw JSON
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data_raw = load_json(train_file_paths["Phase_7_PCR/Train/train.json"])
train_candidates_raw = load_json(train_file_paths["Phase_7_PCR/Train/train_candidates.json"])

dev_data_raw = load_json(dev_file_paths["Phase_7_PCR/Dev/dev.json"])
dev_candidates_raw = load_json(dev_file_paths["Phase_7_PCR/Dev/dev_candidates.json"])

test_data_raw = load_json(test_file_paths["Phase_7_PCR/Test/test.json"])
test_candidates_raw = load_json(test_file_paths["Phase_7_PCR/Test/test_candidates.json"])

print("Train queries:", len(train_data_raw))
print("Train candidates:", len(train_candidates_raw))
print("Dev queries:", len(dev_data_raw))
print("Dev candidates:", len(dev_candidates_raw))
print("Test queries:", len(test_data_raw))
print("Test candidates:", len(test_candidates_raw))


Train queries: 827
Train candidates: 4320
Dev queries: 118
Dev candidates: 1023
Test queries: 237
Test candidates: 1727


In [36]:
class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list


In [37]:
# ------------------------------------------------------------
# Load the existing BPE tokenizer
# ------------------------------------------------------------

BPE_PATH = "/kaggle/input/datasets/ksaiveerendraprasad/tokenizer-results/project_bpe_16k.pkl"

with open(BPE_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("BPE vocab size:", len(tokenizer.itos))

converter = ToTokensConverter(tokenizer.itos)

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3
VOCAB_SIZE = 8000

print("Special IDs:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)


BPE vocab size: 7999
Special IDs: 0 1 2 3


In [38]:
# ------------------------------------------------------------
# Experiment configuration
# ------------------------------------------------------------

MAX_SEQ_LEN = 1024
CHUNK_OVERLAP = 64
CHUNK_CONTENT_LEN = MAX_SEQ_LEN - 2
CHUNK_STRIDE = CHUNK_CONTENT_LEN - CHUNK_OVERLAP

# Match the working reference-style setup.
MAX_CHUNKS = 8
CHUNK_BATCH_SIZE = 8
NUM_NEGATIVES = 4
BATCH_SIZE = 8

EVAL_DOCUMENT_BATCH_SIZE = 4

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TEMPERATURE = 0.07
EPOCHS = 5
GRAD_CLIP = 1.0

CHECKPOINT_DIR = "/kaggle/working/pcr_exp3_lora_random"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best.pt",
)
HISTORY_PATH = os.path.join(
    CHECKPOINT_DIR,
    "history.json",
)
TEST_PREDICTION_PATH = os.path.join(
    CHECKPOINT_DIR,
    "test_predictions_top7.json",
)

print("Experiment:", "Phase-1 Encoder + LoRA + Random Negatives")
print("Adaptation:", "LoRA parameter-efficient adaptation of the Phase-1 attention projections while using random negatives.")
print("MAX_CHUNKS:", MAX_CHUNKS)
print("CHUNK_BATCH_SIZE:", CHUNK_BATCH_SIZE)
print("NUM_NEGATIVES:", NUM_NEGATIVES)
print("BATCH_SIZE:", BATCH_SIZE)


Experiment: Phase-1 Encoder + LoRA + Random Negatives
Adaptation: LoRA parameter-efficient adaptation of the Phase-1 attention projections while using random negatives.
MAX_CHUNKS: 8
CHUNK_BATCH_SIZE: 8
NUM_NEGATIVES: 4
BATCH_SIZE: 8


In [39]:
# ------------------------------------------------------------
# Data processing — same style as the reference notebook
# No normalization is performed.
# ------------------------------------------------------------

def case_text_to_string(text):
    if isinstance(text, str):
        return text

    if isinstance(text, list):
        parts = []
        for item in text:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                raise TypeError(
                    f"Unsupported text item: {type(item)}"
                )
        return " ".join(parts)

    raise TypeError(f"Unsupported text type: {type(text)}")


def prepare_records(raw_records, labeled=False):
    records = {}

    if isinstance(raw_records, dict):
        iterator = raw_records.items()
    elif isinstance(raw_records, list):
        iterator = (
            (
                row.get(
                    "id",
                    row.get("query_id", row.get("case_id")),
                ),
                row,
            )
            for row in raw_records
        )
    else:
        raise TypeError("Dataset must be a dict or list.")

    for raw_id, row in iterator:
        if raw_id is None:
            raise ValueError("A record has no case ID.")

        case_id = str(raw_id)

        if case_id in records:
            raise ValueError(f"Duplicate case ID: {case_id}")

        text = case_text_to_string(row["text"])

        if not text.strip():
            raise ValueError(f"Empty text for case {case_id}")

        item = {"text": text}

        if labeled:
            positives = row["relevant_candidates"]
            if not isinstance(positives, list):
                raise ValueError(
                    "relevant_candidates must be a list."
                )
            item["relevant_candidates"] = [
                str(cid) for cid in positives
            ]

        records[case_id] = item

    return records


# IMPORTANT:
# train_data/dev_data are QUERIES.
# train_candidates/dev_candidates are CANDIDATES.

train_data = prepare_records(
    train_data_raw,
    labeled=True,
)
train_candidates = prepare_records(
    train_candidates_raw,
)

dev_data = prepare_records(
    dev_data_raw,
    labeled=True,
)
dev_candidates = prepare_records(
    dev_candidates_raw,
)

test_data = prepare_records(test_data_raw)
test_candidates = prepare_records(test_candidates_raw)

print("Train:", len(train_data), "queries,", len(train_candidates), "candidates")
print("Dev:  ", len(dev_data), "queries,", len(dev_candidates), "candidates")
print("Test: ", len(test_data), "queries,", len(test_candidates), "candidates")


Train: 827 queries, 4320 candidates
Dev:   118 queries, 1023 candidates
Test:  237 queries, 1727 candidates


In [40]:
# ------------------------------------------------------------
# Dataset integrity checks
# ------------------------------------------------------------

def validate_query_candidate_split(
    query_data,
    candidate_data,
    split_name,
):
    candidate_ids = set(candidate_data)

    positive_counts = []

    for query_id, row in query_data.items():
        positives = set(row["relevant_candidates"])

        missing = positives - candidate_ids
        if missing:
            raise ValueError(
                f"{split_name}/{query_id}: missing candidate IDs "
                f"{sorted(missing)[:10]}"
            )

        if not positives:
            continue
            raise ValueError(
                f"{split_name}/{query_id}: no positive candidates."
            )

        positive_counts.append(len(positives))

    print(
        f"{split_name}: "
        f"{len(query_data)} queries, "
        f"{len(candidate_data)} candidates, "
        f"mean positives/query={np.mean(positive_counts):.3f}"
    )


validate_query_candidate_split(
    train_data,
    train_candidates,
    "TRAIN",
)

validate_query_candidate_split(
    dev_data,
    dev_candidates,
    "DEV",
)

print(
    "Test unlabeled:",
    all(
        "relevant_candidates" not in row
        for row in test_data.values()
    ),
)


TRAIN: 827 queries, 4320 candidates, mean positives/query=6.497
DEV: 118 queries, 1023 candidates, mean positives/query=8.246
Test unlabeled: True


In [41]:
# ------------------------------------------------------------
# Tokenization + chunking — same reference preparation
# ------------------------------------------------------------

def convert_text_to_ids(text):
    token_ids = converter.to_tokens([text])[0]

    if not token_ids:
        return [UNK_ID]

    token_ids = [int(x) for x in token_ids]

    bad = [
        x for x in token_ids
        if x < 0 or x >= VOCAB_SIZE
    ]

    if bad:
        raise ValueError(
            f"Invalid token IDs: {bad[:10]}"
        )

    return token_ids


def document_chunks(text):
    token_ids = convert_text_to_ids(text)

    capacity = MAX_SEQ_LEN - 2
    stride = capacity - CHUNK_OVERLAP

    if len(token_ids) <= capacity:
        chunks = [token_ids]
    else:
        starts = list(
            range(
                0,
                len(token_ids) - capacity + 1,
                stride,
            )
        )

        last_start = len(token_ids) - capacity
        if not starts or starts[-1] != last_start:
            starts.append(last_start)

        chunks = [
            token_ids[start:start + capacity]
            for start in starts
        ]

    if len(chunks) > MAX_CHUNKS:
        positions = np.linspace(
            0,
            len(chunks) - 1,
            MAX_CHUNKS,
        ).round().astype(int)

        chunks = [chunks[i] for i in positions]

    return [
        [BOS_ID] + chunk[:capacity] + [EOS_ID]
        for chunk in chunks
    ]


def prepare_cases(records):
    return {
        case_id: document_chunks(row["text"])
        for case_id, row in records.items()
    }


# Tokenize once. This is CPU-side preprocessing and does not
# allocate the document pool on GPU.
train_query_chunks = prepare_cases(train_data)
train_candidate_chunks = prepare_cases(train_candidates)

dev_query_chunks = prepare_cases(dev_data)
dev_candidate_chunks = prepare_cases(dev_candidates)

test_query_chunks = prepare_cases(test_data)
test_candidate_chunks = prepare_cases(test_candidates)

print("Chunks prepared.")


Chunks prepared.


In [42]:
# ------------------------------------------------------------
# Random negative sampling
# ------------------------------------------------------------
# LoRA + RANDOM negatives.
# No BM25 index is created and no text/vectorization is needed.
# Negatives are sampled once and then cached for reproducibility.
# ------------------------------------------------------------

rng = random.Random(SEED)

train_query_ids = list(train_data.keys())
train_candidate_ids = list(train_candidates.keys())

random_negative_map = {}

for qid in tqdm(
    train_query_ids,
    desc="Sampling random negatives",
):
    positives = list(dict.fromkeys(
        train_data[qid]["relevant_candidates"]
    ))
    positive_set = set(positives)

    eligible = [
        cid for cid in train_candidate_ids
        if cid not in positive_set
    ]

    if len(eligible) < NUM_NEGATIVES:
        raise RuntimeError(
            f"Not enough negatives for {qid}: "
            f"need {NUM_NEGATIVES}, found {len(eligible)}"
        )

    random_negative_map[qid] = rng.sample(
        eligible,
        NUM_NEGATIVES,
    )

print(
    "Random negative sampling complete:",
    len(random_negative_map),
    "queries",
)

example_qid = train_query_ids[0]
print("Example query:", example_qid)
print("Random negatives:", random_negative_map[example_qid])

RANDOM_NEGATIVE_CACHE_PATH = os.path.join(
    CHECKPOINT_DIR,
    "random_train_negative_map.json",
)

with open(RANDOM_NEGATIVE_CACHE_PATH, "w", encoding="utf-8") as f:
    json.dump(random_negative_map, f, indent=2)

print("Saved:", RANDOM_NEGATIVE_CACHE_PATH)


Sampling random negatives:   0%|          | 0/827 [00:00<?, ?it/s]

Random negative sampling complete: 827 queries
Example query: 0000000599
Random negatives: ['0000432540', '0000088171', '0001069919', '0000952560']
Saved: /kaggle/working/pcr_exp3_lora_random/random_train_negative_map.json


In [43]:
# ------------------------------------------------------------
# PCR Dataset — random negatives
# ------------------------------------------------------------

class PCRDataset(Dataset):
    def __init__(
        self,
        query_data,
        candidate_data,
        negative_map,
        num_negatives=4,
    ):
        self.query_data = query_data
        self.candidate_ids = list(candidate_data)
        self.negative_map = negative_map
        self.num_negatives = num_negatives

        self.query_ids = [
            qid for qid, row in query_data.items()
            if row["relevant_candidates"]
        ]

    def __len__(self):
        return len(self.query_ids)

    def __getitem__(self, idx):
        qid = self.query_ids[idx]

        positives = list(dict.fromkeys(
            self.query_data[qid]["relevant_candidates"]
        ))

        negatives = list(self.negative_map[qid])

        if len(negatives) != self.num_negatives:
            raise ValueError(
                f"{qid}: expected {self.num_negatives} negatives, "
                f"got {len(negatives)}"
            )

        if set(positives) & set(negatives):
            raise ValueError(
                f"{qid}: a sampled negative is a positive candidate."
            )

        return {
            "query_id": qid,
            "positive_candidate_ids": positives,
            "negative_candidate_ids": negatives,
        }

train_dataset = PCRDataset(
    train_data,
    train_candidates,
    random_negative_map,
    num_negatives=NUM_NEGATIVES,
)

print("Train dataset size:", len(train_dataset))

# ------------------------------------------------------------
# Training DataLoader
# ------------------------------------------------------------

def pcr_collate_fn(batch):
    return batch

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    collate_fn=pcr_collate_fn,
)

print("Train dataset size:", len(train_dataset))
print("Train loader batches:", len(train_loader))
print("Train loader batch size:", train_loader.batch_size)


Train dataset size: 817
Train dataset size: 817
Train loader batches: 103
Train loader batch size: 8


In [44]:
# ------------------------------------------------------------
# Lightweight batch sanity check
# ------------------------------------------------------------

sample = next(iter(train_loader))[0]

print("Query:", sample["query_id"])
print("Number of positives:", len(sample["positive_candidate_ids"]))
print("Number of negatives:", len(sample["negative_candidate_ids"]))

assert len(sample["negative_candidate_ids"]) == NUM_NEGATIVES


Query: 0001207302
Number of positives: 4
Number of negatives: 4


In [45]:
# ------------------------------------------------------------
# No GPU tensors are created by the DataLoader.
# This is intentional and keeps preprocessing memory-safe.
# ------------------------------------------------------------

print("DataLoader preparation complete.")
print("GPU tensors are created only inside encode_one_document().")


DataLoader preparation complete.
GPU tensors are created only inside encode_one_document().


In [46]:
# ------------------------------------------------------------
# Lightweight loader sanity check
# ------------------------------------------------------------

sample_batch = next(iter(train_loader))
assert isinstance(sample_batch, list)
assert len(sample_batch) == BATCH_SIZE

sample = sample_batch[0]

print("Query:", sample["query_id"])
print("Positive candidates:", len(sample["positive_candidate_ids"]))
print("Random negative candidates:", len(sample["negative_candidate_ids"]))
print("Random negatives:", sample["negative_candidate_ids"])

assert len(sample["negative_candidate_ids"]) == NUM_NEGATIVES


Query: 0000645052
Positive candidates: 4
Random negative candidates: 4
Random negatives: ['0000578327', '0000009324', '0001078597', '0000566292']


In [47]:
# ------------------------------------------------------------
# RoPE
# ------------------------------------------------------------

def build_rope_cache(
    max_seq_len: int,
    d_head: int,
    base: float = 10000.0,
):
    assert d_head % 2 == 0

    inverse_frequencies = torch.tensor(
        [
            1 / base ** (2 * i / d_head)
            for i in range(d_head // 2)
        ],
        dtype=torch.float32,
    )

    positions = torch.arange(
        max_seq_len,
        dtype=torch.float32,
    )

    freqs = torch.outer(
        positions,
        inverse_frequencies,
    )

    angles = torch.cat([freqs, freqs], dim=-1)

    return angles.cos(), angles.sin()


def rotate_half(x):
    d_head = x.shape[-1]
    x1 = x[..., :d_head // 2]
    x2 = x[..., d_head // 2:]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return x * cos + rotate_half(x) * sin


In [48]:
# ------------------------------------------------------------
# Attention + encoder architecture
# ------------------------------------------------------------

def build_attn_mask(is_causal, padding_mask, T):
    """
    padding_mask=True means VALID token.
    Returned bool mask=True means attention is allowed.
    """
    full = torch.ones(
        T, T,
        dtype=torch.bool,
        device=is_causal.device,
    )

    causal_allow = torch.tril(full)

    allow = torch.where(
        is_causal[:, None, None, None],
        causal_allow,
        full,
    )

    if padding_mask is not None:
        valid_keys = padding_mask.to(torch.bool)
        allow = allow & valid_keys[:, None, None, :]

    return allow


class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        max_seq_len,
        dropout=0.0,
        use_rope=False,
        attention_scale=None,
    ):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope

        self.attention_scale = (
            1.0 / math.sqrt(self.d_head)
            if attention_scale is None
            else float(attention_scale)
        )

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.d_head,
            bias=False,
        )
        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.d_head,
            bias=False,
        )
        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.d_head,
            bias=False,
        )
        self.out_proj = nn.Linear(
            n_heads * self.d_head,
            d_model,
            bias=False,
        )

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if use_rope:
            cos, sin = build_rope_cache(
                max_seq_len,
                self.d_head,
            )
            self.register_buffer(
                "rope_cos",
                cos,
                persistent=False,
            )
            self.register_buffer(
                "rope_sin",
                sin,
                persistent=False,
            )

    def forward(self, x, attn_mask=None):
        B, T, C = x.shape

        q = self.q_proj(x).reshape(
            B, T, self.n_heads, self.d_head
        ).transpose(1, 2)

        k = self.k_proj(x).reshape(
            B, T, self.n_kv_heads, self.d_head
        ).transpose(1, 2)

        v = self.v_proj(x).reshape(
            B, T, self.n_kv_heads, self.d_head
        ).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(
                q,
                self.rope_cos[:T],
                self.rope_sin[:T],
            )
            k = apply_rope(
                k,
                self.rope_cos[:T],
                self.rope_sin[:T],
            )

        if self.n_rep > 1:
            k = k.repeat_interleave(
                self.n_rep, dim=1
            )
            v = v.repeat_interleave(
                self.n_rep, dim=1
            )

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.attn_dropout.p
                if self.training else 0.0
            ),
            scale=self.attention_scale,
        )

        out = out.transpose(1, 2).reshape(
            B, T, C
        )

        out = self.out_proj(out)
        out = self.resid_dropout(out)

        return out


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 10
    n_heads: int = 8
    n_kv_heads: int = 8
    d_ff: int = 4096
    max_seq_len: int = 1024
    dropout: float = 0.1
    pos_emb_type: str = "rope"
    attention_scale: float = None


class EncoderFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.ln1 = nn.LayerNorm(cfg.d_model)

        self.attn = MultiHeadAttention(
            cfg.d_model,
            cfg.n_heads,
            cfg.n_kv_heads,
            cfg.max_seq_len,
            cfg.dropout,
            use_rope=(cfg.pos_emb_type == "rope"),
            attention_scale=cfg.attention_scale,
        )

        self.ln2 = nn.LayerNorm(cfg.d_model)

        self.ffn = EncoderFeedForward(
            cfg.d_model,
            cfg.d_ff,
            cfg.dropout,
        )

    def forward(self, x, attn_mask=None):
        x = x + self.attn(
            self.ln1(x),
            attn_mask,
        )
        x = x + self.ffn(
            self.ln2(x)
        )
        return x


class EncoderLM(nn.Module):
    def __init__(
        self,
        cfg,
        gradient_checkpointing=True,
    ):
        super().__init__()

        self.cfg = cfg
        self.gradient_checkpointing = gradient_checkpointing

        self.tok_emb = nn.Embedding(
            cfg.vocab_size,
            cfg.d_model,
        )

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(
                cfg.max_seq_len,
                cfg.d_model,
            )
            self.register_buffer(
                "pos_emb",
                pe,
                persistent=False,
            )

        self.blocks = nn.ModuleList(
            [
                EncoderBlock(cfg)
                for _ in range(cfg.n_layers)
            ]
        )

        self.ln_f = nn.LayerNorm(cfg.d_model)

        self.lm_head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.tok_emb.weight

    def forward(
        self,
        idx,
        padding_mask=None,
        is_causal=None,
    ):
        B, T = idx.shape

        if T > self.cfg.max_seq_len:
            raise ValueError(
                f"T={T} > max_seq_len={self.cfg.max_seq_len}"
            )

        if padding_mask is not None:
            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool,
            )

        if is_causal is None:
            is_causal = torch.zeros(
                B,
                dtype=torch.bool,
                device=idx.device,
            )

        attn_mask = build_attn_mask(
            is_causal,
            padding_mask,
            T,
        )

        x = self.tok_emb(idx)

        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T].unsqueeze(0)

        for block in self.blocks:
            x = block(
                x,
                attn_mask=attn_mask,
            )

        x = self.ln_f(x)

        return self.lm_head(x)

    def encode(self, idx, padding_mask=None):
        """
        Bidirectional encoder path used by PCR.

        Gradient checkpointing is applied one Transformer block
        at a time during training. This is the main OOM fix.
        """
        B, T = idx.shape

        if T > self.cfg.max_seq_len:
            raise ValueError(
                f"T={T} > max_seq_len={self.cfg.max_seq_len}"
            )

        if padding_mask is not None:
            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool,
            )

        is_causal = torch.zeros(
            B,
            dtype=torch.bool,
            device=idx.device,
        )

        attn_mask = build_attn_mask(
            is_causal,
            padding_mask,
            T,
        )

        x = self.tok_emb(idx)

        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T].unsqueeze(0)

        for block in self.blocks:
            if (
                self.training
                and self.gradient_checkpointing
            ):
                def run_block(
                    hidden,
                    block=block,
                    attn_mask=attn_mask,
                ):
                    return block(
                        hidden,
                        attn_mask=attn_mask,
                    )

                x = checkpoint(
                    run_block,
                    x,
                    use_reentrant=False,
                )
            else:
                x = block(
                    x,
                    attn_mask=attn_mask,
                )

        x = self.ln_f(x)

        return x


In [49]:
# ------------------------------------------------------------
# Instantiate and load Phase-1 checkpoint
# ------------------------------------------------------------

MODEL_DIM = 512
MODEL_LAYERS = 10
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8

cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="rope",
)

model_phase1 = EncoderLM(
    cfg,
    gradient_checkpointing=True,
)

PHASE1_CHECKPOINT = (
    "/kaggle/input/datasets/ksaiveerendraprasad/hybrid1m/hybrid_encoder.pt"
)

checkpoint_data = torch.load(
    PHASE1_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

state_dict = checkpoint_data["model_state_dict"]

clean_state_dict = {}
for key, value in state_dict.items():
    if key.startswith("module."):
        key = key[len("module."):]
    clean_state_dict[key] = value

missing, unexpected = model_phase1.load_state_dict(
    clean_state_dict,
    strict=True,
)

print("Phase-1 checkpoint loaded successfully.")

model_phase1 = model_phase1.to(device)

print(
    "Trainable parameters:",
    sum(p.numel() for p in model_phase1.parameters() if p.requires_grad),
)


Phase-1 checkpoint loaded successfully.
Trainable parameters: 56592384


In [51]:
# ------------------------------------------------------------
# LoRA adaptation
#
# LoRA is multiplicative rather than additive LoRA.
# We use the additive rank-inflation variant (LoRA):
#
#   h = (I + alpha/r * B A) W0 x
#     = W0 x + alpha/r * B A (W0 x)
#
# The base Phase-1 weights are frozen.
# A is trainable and initialized randomly; B is zero-initialized,
# so the adapter starts exactly at the Phase-1 model.
#
# Applied to q_proj, k_proj, v_proj, and out_proj in every
# Transformer block. These attention projections are square
# 512 x 512 matrices in this Phase-1 architecture.
# ------------------------------------------------------------

class LoRALinear(nn.Module):
    def __init__(
        self,
        layer,
        rank=16,
        alpha=32,
        dropout=0.05,
    ):
        super().__init__()

        if not isinstance(layer, nn.Linear):
            raise TypeError(
                "LoRALinear expects nn.Linear"
            )

        if layer.in_features != layer.out_features:
            raise ValueError(
                "This LoRA implementation expects square "
                "projection matrices."
            )

        self.layer = layer
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank
        self.dropout = nn.Dropout(dropout)

        for p in self.layer.parameters():
            p.requires_grad_(False)

        d = layer.in_features

        # A: [rank, d]
        self.A = nn.Parameter(
            torch.empty(
                self.rank,
                d,
            )
        )

        # B: [d, rank], zero initialized so the initial
        # multiplicative update is exactly identity.
        self.B = nn.Parameter(
            torch.zeros(
                d,
                self.rank,
            )
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=5**0.5,
        )

    def forward(self, x):
        base_output = self.layer(x)

        low_rank = F.linear(
            self.dropout(base_output),
            self.A,
        )

        multiplicative_update = F.linear(
            low_rank,
            self.B,
        )

        return (
            base_output
            + self.scaling * multiplicative_update
        )


LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


# Load the Phase-1 checkpoint first.
model_phase1 = EncoderLM(
    cfg,
    gradient_checkpointing=True,
)

PHASE1_CHECKPOINT = (
    "/kaggle/input/datasets/ksaiveerendraprasad/hybrid1m/hybrid_encoder.pt"
)

checkpoint_data = torch.load(
    PHASE1_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

state_dict = checkpoint_data["model_state_dict"]

clean_state_dict = {}
for key, value in state_dict.items():
    if key.startswith("module."):
        key = key[len("module."):]
    clean_state_dict[key] = value

model_phase1.load_state_dict(
    clean_state_dict,
    strict=True,
)

print("Phase-1 checkpoint loaded successfully.")


# Freeze the complete Phase-1 encoder.
for p in model_phase1.parameters():
    p.requires_grad_(False)


# Inject LoRA into the attention projections.
target_names = (
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
)

for block in model_phase1.blocks:
    for name in target_names:
        old_layer = getattr(block.attn, name)

        setattr(
            block.attn,
            name,
            LoRALinear(
                old_layer,
                rank=LORA_RANK,
                alpha=LORA_ALPHA,
                dropout=LORA_DROPOUT,
            ),
        )


model_phase1 = model_phase1.to(device)


class EncoderWrapper(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, input_ids, padding_mask):
        return self.encoder.encode(
            input_ids,
            padding_mask=padding_mask,
        )


encoder_model = EncoderWrapper(
    model_phase1
).to(device)


trainable = sum(
    p.numel()
    for p in encoder_model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in encoder_model.parameters()
)

print("Total parameters:", total)
print("Trainable parameters:", trainable)
print(
    "Trainable percentage:",
    100.0 * trainable / total,
)


Phase-1 checkpoint loaded successfully.
Total parameters: 57247744
Trainable parameters: 655360
Trainable percentage: 1.1447787357349837


In [52]:
# ------------------------------------------------------------
# Optimizer + AMP scaler
# ------------------------------------------------------------

trainable_parameters = [
    p
    for p in encoder_model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda"),
)

print(
    "Trainable tensors:",
    len(trainable_parameters),
)
print("Optimizer and GradScaler ready.")


Trainable tensors: 80
Optimizer and GradScaler ready.


In [53]:
# ------------------------------------------------------------
# VRAM report
# ------------------------------------------------------------

def gpu_memory_report(prefix=""):
    if not torch.cuda.is_available():
        print("CUDA unavailable.")
        return

    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        allocated = torch.cuda.memory_allocated(i)
        reserved = torch.cuda.memory_reserved(i)
        peak = torch.cuda.max_memory_allocated(i)

        print(
            f"{prefix}GPU {i}: "
            f"free={free/1024**3:.2f} GB, "
            f"allocated={allocated/1024**3:.2f} GB, "
            f"reserved={reserved/1024**3:.2f} GB, "
            f"peak={peak/1024**3:.2f} GB"
        )

gpu_memory_report("After model setup | ")


After model setup | GPU 0: free=14.24 GB, allocated=0.22 GB, reserved=0.22 GB, peak=0.22 GB
After model setup | GPU 1: free=14.46 GB, allocated=0.00 GB, reserved=0.00 GB, peak=0.00 GB


In [54]:
# ------------------------------------------------------------
# Hierarchical document encoding
# ------------------------------------------------------------

def chunks_to_tensors(chunks):
    lengths = torch.tensor(
        [len(chunk) for chunk in chunks],
        dtype=torch.long,
        device=device,
    )

    input_ids = torch.full(
        (len(chunks), MAX_SEQ_LEN),
        PAD_ID,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (len(chunks), MAX_SEQ_LEN),
        dtype=torch.bool,
    )

    for i, chunk in enumerate(chunks):
        length = min(len(chunk), MAX_SEQ_LEN)
        input_ids[i, :length] = torch.tensor(
            chunk[:length],
            dtype=torch.long,
        )
        attention_mask[i, :length] = True

    return (
        input_ids.to(device, non_blocking=True),
        attention_mask.to(device, non_blocking=True),
    )


def encode_one_document(
    model,
    chunks,
    chunk_batch_size=2,
):
    """
    Hierarchical mean pooling:

        token hidden states
          -> mean over valid tokens
        chunk embeddings
          -> mean over chunks
        L2 normalization
    """
    pooled_chunks = []

    for start in range(
        0,
        len(chunks),
        chunk_batch_size,
    ):
        part = chunks[start:start + chunk_batch_size]

        input_ids, attention_mask = chunks_to_tensors(part)

        hidden_states = model(
            input_ids,
            padding_mask=attention_mask,
        )

        token_weights = attention_mask.unsqueeze(-1).to(
            hidden_states.dtype
        )

        chunk_embeddings = (
            (hidden_states * token_weights).sum(dim=1)
            / token_weights.sum(dim=1).clamp(min=1.0)
        )

        pooled_chunks.append(chunk_embeddings)

        del input_ids, attention_mask, hidden_states
        del token_weights, chunk_embeddings

    pooled_chunks = torch.cat(pooled_chunks, dim=0)

    document_embedding = pooled_chunks.mean(dim=0)

    return F.normalize(
        document_embedding.float(),
        p=2,
        dim=0,
    )


def encode_document_list(
    model,
    documents,
    chunk_batch_size=2,
):
    embeddings = []

    for chunks in documents:
        embeddings.append(
            encode_one_document(
                model,
                chunks,
                chunk_batch_size=chunk_batch_size,
            )
        )

    return torch.stack(embeddings, dim=0)


In [55]:
# ------------------------------------------------------------
# Multi-positive contrastive loss
# ------------------------------------------------------------

def multi_positive_contrastive_loss(
    similarity_scores,
    positive_mask,
    temperature=0.07,
):
    logits = similarity_scores / temperature

    positive_logits = logits.masked_fill(
        ~positive_mask,
        float("-inf"),
    )

    if not positive_mask.any(dim=1).all():
        raise ValueError(
            "Every query must have at least one positive."
        )

    numerator = torch.logsumexp(
        positive_logits,
        dim=1,
    )

    denominator = torch.logsumexp(
        logits,
        dim=1,
    )

    return -(numerator - denominator).mean()


In [56]:
# ------------------------------------------------------------
# One-batch DATA sanity check only
# ------------------------------------------------------------

batch = next(iter(train_loader))
item = batch[0]

candidate_ids = list(
    dict.fromkeys(
        item["positive_candidate_ids"]
        + item["negative_candidate_ids"]
    )
)

print("Query:", item["query_id"])
print("Positive candidates:", len(item["positive_candidate_ids"]))
print("Negative candidates:", len(item["negative_candidate_ids"]))
print("Unique candidates this step:", len(candidate_ids))

assert len(item["negative_candidate_ids"]) == NUM_NEGATIVES

print("No GPU model forward was performed in this sanity check.")


Query: 0000672592
Positive candidates: 5
Negative candidates: 4
Unique candidates this step: 9
No GPU model forward was performed in this sanity check.


In [57]:
# ------------------------------------------------------------
# Training
#
# Reference-style lazy data flow:
#   DataLoader -> query IDs
#   -> sample candidates
#   -> pre-tokenized/chunked CPU lists
#   -> only current document chunks go to GPU
#
# Additionally, save_on_cpu() offloads tensors saved for backward.
# This is the main memory safeguard for FULL Phase-1 fine-tuning.
# ------------------------------------------------------------

def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scaler,
    device,
    temperature=0.07,
    chunk_batch_size=2,
    grad_clip=1.0,
):
    model.train()

    total_loss = 0.0
    total_queries = 0

    progress = tqdm(
        train_loader,
        desc="Training",
        leave=False,
    )

    for step, batch in enumerate(progress, 1):
        optimizer.zero_grad(set_to_none=True)

        query_id = batch[0]["query_id"]

        positive_ids = batch[0]["positive_candidate_ids"]
        negative_ids = batch[0]["negative_candidate_ids"]

        candidate_ids = list(
            dict.fromkeys(
                positive_ids + negative_ids
            )
        )

        try:
            with torch.autograd.graph.save_on_cpu(
                pin_memory=True
            ):
                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.float16,
                    enabled=(device.type == "cuda"),
                ):
                    query_embedding = encode_one_document(
                        model,
                        train_query_chunks[query_id],
                        chunk_batch_size,
                    )

                    candidate_embeddings = encode_document_list(
                        model,
                        [
                            train_candidate_chunks[cid]
                            for cid in candidate_ids
                        ],
                        chunk_batch_size,
                    )

                    scores = (
                        query_embedding.unsqueeze(0)
                        @ candidate_embeddings.T
                    )

                    positive_mask = torch.tensor(
                        [
                            cid in set(positive_ids)
                            for cid in candidate_ids
                        ],
                        dtype=torch.bool,
                        device=device,
                    ).unsqueeze(0)

                    loss = multi_positive_contrastive_loss(
                        scores,
                        positive_mask,
                        temperature,
                    )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Non-finite retrieval loss."
                )

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                grad_clip,
            )

            scaler.step(optimizer)
            scaler.update()

        except torch.cuda.OutOfMemoryError:
            print(
                f"\nCUDA OOM at training step {step}."
            )
            clear_cuda_memory()
            raise

        loss_value = loss.detach().float().item()

        total_loss += loss_value
        total_queries += 1

        progress.set_postfix(
            loss=f"{loss_value:.4f}"
        )

        del (
            query_embedding,
            candidate_embeddings,
            scores,
            positive_mask,
            loss,
        )

        clear_cuda_memory()

    return total_loss / max(total_queries, 1)


In [58]:
# ------------------------------------------------------------
# Full-pool evaluation utilities
# ------------------------------------------------------------

@torch.no_grad()
def encode_record_pool(
    model,
    records,
    batch_size=4,
    chunk_batch_size=2,
    desc="Encoding pool",
):
    model.eval()

    ids = list(records.keys())
    all_embeddings = []

    for start in tqdm(
        range(0, len(ids), batch_size),
        desc=desc,
    ):
        batch_ids = ids[start:start + batch_size]

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            batch_embeddings = encode_document_list(
                model,
                [records[cid] for cid in batch_ids],
                chunk_batch_size,
            )

        all_embeddings.append(
            batch_embeddings.float().cpu()
        )

        del batch_embeddings

        # No graph exists because of @torch.no_grad().
        clear_cuda_memory()

    return ids, torch.cat(all_embeddings, dim=0)


@torch.no_grad()
def evaluate_retrieval_full_pool(
    model,
    query_records,
    candidate_records,
    query_data,
    k=7,
    eval_batch_size=4,
    chunk_batch_size=2,
):
    model.eval()

    candidate_ids, candidate_embeddings = encode_record_pool(
        model,
        candidate_records,
        batch_size=eval_batch_size,
        chunk_batch_size=chunk_batch_size,
        desc="Encoding FULL candidate pool",
    )

    query_ids, query_embeddings = encode_record_pool(
        model,
        query_records,
        batch_size=eval_batch_size,
        chunk_batch_size=chunk_batch_size,
        desc="Encoding ALL dev queries",
    )

    total_tp = 0
    total_fp = 0
    total_fn = 0

    predictions = {}
    top_k = min(k, len(candidate_ids))

    for start in range(0, len(query_ids), 64):
        qids = query_ids[start:start + 64]
        qvec = query_embeddings[start:start + 64]

        scores = qvec @ candidate_embeddings.T

        indices = torch.topk(
            scores,
            k=top_k,
            dim=1,
        ).indices

        for local_i, qid in enumerate(qids):
            ranked = [
                candidate_ids[j]
                for j in indices[local_i].tolist()
            ]

            predictions[qid] = ranked

            gold = set(
                query_data[qid]["relevant_candidates"]
            )
            predicted = set(ranked)

            total_tp += len(gold & predicted)
            total_fp += len(predicted - gold)
            total_fn += len(gold - predicted)

        del scores, indices

    precision = total_tp / max(
        total_tp + total_fp,
        1,
    )
    recall = total_tp / max(
        total_tp + total_fn,
        1,
    )
    f1 = (
        2 * precision * recall
        / max(precision + recall, 1e-12)
    )

    metrics = {
        f"Micro-Precision@{k}": precision,
        f"Micro-Recall@{k}": recall,
        f"Micro-F1@{k}": f1,
        "num_queries": len(query_ids),
        "num_candidates": len(candidate_ids),
        "total_tp": total_tp,
        "total_fp": total_fp,
        "total_fn": total_fn,
    }

    del candidate_embeddings
    del query_embeddings
    clear_cuda_memory()

    return metrics, predictions


dev_query_records = dev_query_chunks
dev_candidate_records = dev_candidate_chunks
test_query_records = test_query_chunks
test_candidate_records = test_candidate_chunks


In [59]:
# ------------------------------------------------------------
# Evaluation preparation check
# ------------------------------------------------------------

print(
    "Dev query records:",
    len(dev_query_records),
)
print(
    "Dev candidate records:",
    len(dev_candidate_records),
)


Dev query records: 118
Dev candidate records: 1023


In [60]:
# ------------------------------------------------------------
# Full-pool evaluation function is defined above.
# ------------------------------------------------------------

print("Full-pool evaluation ready.")


Full-pool evaluation ready.


In [ ]:
# ------------------------------------------------------------
# Check full-pool DEV evaluation once before training
# ------------------------------------------------------------

# This is optional but useful to confirm the evaluation path
# and memory behavior before spending time on training.

baseline_dev_metrics, baseline_dev_predictions = (
    evaluate_retrieval_full_pool(
        model=encoder_model,
        query_records=dev_query_records,
        candidate_records=dev_candidate_records,
        query_data=dev_data,
        k=7,
        eval_batch_size=EVAL_DOCUMENT_BATCH_SIZE,
        chunk_batch_size=CHUNK_BATCH_SIZE,
    )
)

print(json.dumps(
    baseline_dev_metrics,
    indent=2,
))


In [62]:
# ------------------------------------------------------------
# Checkpoint helpers
# ------------------------------------------------------------

def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scaler,
    train_loss,
    metrics=None,
):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": train_loss,
        "metrics": metrics,
        "config": {
            "max_seq_len": MAX_SEQ_LEN,
            "chunk_overlap": CHUNK_OVERLAP,
            "max_chunks": MAX_CHUNKS,
            "num_negatives": NUM_NEGATIVES,
            "chunk_batch_size": CHUNK_BATCH_SIZE,
            "temperature": TEMPERATURE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "gradient_checkpointing": True,
            "data_parallel": False,
            "adaptation": "LoRA + random negatives",
        },
    }

    torch.save(payload, path)
    return path


In [64]:
EPOCHS = 10

In [65]:
# ------------------------------------------------------------
# Full training loop
# ------------------------------------------------------------

best_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    print()
    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("=" * 80)

    clear_cuda_memory()
    gpu_memory_report("Before train | ")

    train_loss = train_one_epoch(
        model=encoder_model,
        train_loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
        temperature=TEMPERATURE,
        chunk_batch_size=CHUNK_BATCH_SIZE,
        grad_clip=GRAD_CLIP,
    )

    print(f"Train loss: {train_loss:.6f}")
    gpu_memory_report("After train | ")

    epoch_checkpoint = os.path.join(
        CHECKPOINT_DIR,
        f"epoch_{epoch}.pt",
    )

    save_checkpoint(
        epoch_checkpoint,
        epoch,
        encoder_model,
        optimizer,
        scaler,
        train_loss,
        metrics=None,
    )

    print("Saved:", epoch_checkpoint)

    dev_metrics, dev_predictions = evaluate_retrieval_full_pool(
        model=encoder_model,
        query_records=dev_query_records,
        candidate_records=dev_candidate_records,
        query_data=dev_data,
        k=7,
        eval_batch_size=EVAL_DOCUMENT_BATCH_SIZE,
        chunk_batch_size=CHUNK_BATCH_SIZE,
    )

    print(
        f"Micro-Precision@7: "
        f"{dev_metrics['Micro-Precision@7']:.6f}"
    )
    print(
        f"Micro-Recall@7: "
        f"{dev_metrics['Micro-Recall@7']:.6f}"
    )
    print(
        f"Micro-F1@7: "
        f"{dev_metrics['Micro-F1@7']:.6f}"
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **dev_metrics,
    }
    history.append(row)

    with open(
        HISTORY_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(history, f, indent=2)

    current_f1 = dev_metrics["Micro-F1@7"]

    if current_f1 > best_f1:
        best_f1 = current_f1

        save_checkpoint(
            BEST_CHECKPOINT,
            epoch,
            encoder_model,
            optimizer,
            scaler,
            train_loss,
            metrics=dev_metrics,
        )

        print(
            "Saved BEST checkpoint:",
            BEST_CHECKPOINT,
        )

    clear_cuda_memory()
    gpu_memory_report("After epoch | ")

print()
print("Training complete.")
print("Best Micro-F1@7:", best_f1)



Epoch 1/10
Before train | GPU 0: free=14.15 GB, allocated=0.27 GB, reserved=0.28 GB, peak=0.27 GB
Before train | GPU 1: free=14.46 GB, allocated=0.00 GB, reserved=0.00 GB, peak=0.00 GB


Training:   0%|          | 0/103 [00:00<?, ?it/s]

Train loss: 0.357488
After train | GPU 0: free=14.09 GB, allocated=0.28 GB, reserved=0.31 GB, peak=0.28 GB
After train | GPU 1: free=14.44 GB, allocated=0.00 GB, reserved=0.00 GB, peak=0.00 GB
Saved: /kaggle/working/pcr_exp3_lora_random/epoch_1.pt


Encoding FULL candidate pool:   0%|          | 0/256 [00:00<?, ?it/s]

Encoding ALL dev queries:   0%|          | 0/30 [00:00<?, ?it/s]

Micro-Precision@7: 0.240920
Micro-Recall@7: 0.204522
Micro-F1@7: 0.221234
Saved BEST checkpoint: /kaggle/working/pcr_exp3_lora_random/best.pt
After epoch | GPU 0: free=14.09 GB, allocated=0.28 GB, reserved=0.31 GB, peak=0.28 GB
After epoch | GPU 1: free=14.44 GB, allocated=0.00 GB, reserved=0.00 GB, peak=0.00 GB

Epoch 2/10
Before train | GPU 0: free=14.09 GB, allocated=0.28 GB, reserved=0.31 GB, peak=0.28 GB
Before train | GPU 1: free=14.44 GB, allocated=0.00 GB, reserved=0.00 GB, peak=0.00 GB


... (41 outputs left out)

Train loss: 0.107797
After train | GPU 0: free=14.09 GB, allocated=0.28 GB, reserved=0.31 GB, peak=0.28 GB
After train | GPU 1: free=14.44 GB, allocated=0.00 GB, reserved=0.00 GB, peak=0.00 GB
Saved: /kaggle/working/pcr_exp3_lora_random/epoch_10.pt


Encoding FULL candidate pool:   0%|          | 0/256 [00:00<?, ?it/s]

Encoding ALL dev queries:   0%|          | 0/30 [00:00<?, ?it/s]

Micro-Precision@7: 0.297821
Micro-Recall@7: 0.252826
Micro-F1@7: 0.273485
Saved BEST checkpoint: /kaggle/working/pcr_exp3_lora_random/best.pt
After epoch | GPU 0: free=14.09 GB, allocated=0.28 GB, reserved=0.31 GB, peak=0.28 GB
After epoch | GPU 1: free=14.44 GB, allocated=0.00 GB, reserved=0.00 GB, peak=0.00 GB

Training complete.
Best Micro-F1@7: 0.273485269594219


In [66]:
# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

history_df = pd.DataFrame(history)

if len(history_df):
    display(history_df)

    history_df.to_csv(
        os.path.join(
            CHECKPOINT_DIR,
            "training_history.csv",
        ),
        index=False,
    )
else:
    print("No validation history available.")


,epoch,train_loss,Micro-Precision@7,Micro-Recall@7,Micro-F1@7,num_queries,num_candidates,total_tp,total_fp,total_fn
0,1,0.357488,0.240920,0.204522,0.221234,118,1023,199,627,774
1,2,0.198251,0.260291,0.220966,0.239022,118,1023,215,611,758
2,3,0.214741,0.282082,0.239466,0.259033,118,1023,233,593,740
3,4,0.149182,0.283293,0.240493,0.260145,118,1023,234,592,739
4,5,0.189483,0.292978,0.248715,0.269038,118,1023,242,584,731
5,6,0.352658,0.291768,0.247688,0.267927,118,1023,241,585,732
6,7,0.266577,0.291768,0.247688,0.267927,118,1023,241,585,732
7,8,0.398471,0.290557,0.246660,0.266815,118,1023,240,586,733
8,9,0.175433,0.296610,0.251799,0.272374,118,1023,245,581,728
9,10,0.107797,0.297821,0.252826,0.273485,118,1023,246,580,727


In [67]:
# ------------------------------------------------------------
# Reload BEST checkpoint
# ------------------------------------------------------------

best_checkpoint_data = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

encoder_model.load_state_dict(
    best_checkpoint_data["model_state_dict"],
    strict=True,
)

encoder_model.eval()

print(
    "Loaded best checkpoint from epoch:",
    best_checkpoint_data["epoch"],
)
print(
    "Best stored metrics:",
    best_checkpoint_data.get("metrics"),
)


Loaded best checkpoint from epoch: 10
Best stored metrics: {'Micro-Precision@7': 0.29782082324455206, 'Micro-Recall@7': 0.2528263103802672, 'Micro-F1@7': 0.273485269594219, 'num_queries': 118, 'num_candidates': 1023, 'total_tp': 246, 'total_fp': 580, 'total_fn': 727}


In [68]:


clear_cuda_memory()

final_dev_metrics, final_dev_predictions = (
    evaluate_retrieval_full_pool(
        model=encoder_model,
        query_records=dev_query_records,
        candidate_records=dev_candidate_records,
        query_data=dev_data,
        k=7,
        eval_batch_size=EVAL_DOCUMENT_BATCH_SIZE,
        chunk_batch_size=CHUNK_BATCH_SIZE,
    )
)

print(json.dumps(final_dev_metrics, indent=2))


Encoding FULL candidate pool:   0%|          | 0/256 [00:00<?, ?it/s]

Encoding ALL dev queries:   0%|          | 0/30 [00:00<?, ?it/s]

{
  "Micro-Precision@7": 0.29782082324455206,
  "Micro-Recall@7": 0.2528263103802672,
  "Micro-F1@7": 0.273485269594219,
  "num_queries": 118,
  "num_candidates": 1023,
  "total_tp": 246,
  "total_fp": 580,
  "total_fn": 727
}


In [69]:


@torch.no_grad()
def generate_test_predictions(
    model,
    query_records,
    candidate_records,
    k=7,
    eval_batch_size=4,
    chunk_batch_size=2,
):
    model.eval()

    candidate_ids, candidate_embeddings = encode_record_pool(
        model,
        candidate_records,
        batch_size=eval_batch_size,
        chunk_batch_size=chunk_batch_size,
        desc="Encoding TEST candidate pool",
    )

    query_ids, query_embeddings = encode_record_pool(
        model,
        query_records,
        batch_size=eval_batch_size,
        chunk_batch_size=chunk_batch_size,
        desc="Encoding TEST queries",
    )

    predictions = {}
    top_k = min(k, len(candidate_ids))

    for start in range(0, len(query_ids), 64):
        qids = query_ids[start:start + 64]
        scores = (
            query_embeddings[start:start + 64]
            @ candidate_embeddings.T
        )

        indices = torch.topk(
            scores,
            k=top_k,
            dim=1,
        ).indices

        for local_i, qid in enumerate(qids):
            predictions[qid] = [
                candidate_ids[j]
                for j in indices[local_i].tolist()
            ]

        del scores, indices

    del candidate_embeddings
    del query_embeddings
    clear_cuda_memory()

    return predictions


test_predictions = generate_test_predictions(
    encoder_model,
    test_query_records,
    test_candidate_records,
    k=7,
    eval_batch_size=EVAL_DOCUMENT_BATCH_SIZE,
    chunk_batch_size=CHUNK_BATCH_SIZE,
)

with open(
    TEST_PREDICTION_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(test_predictions, f, indent=2)

print("Saved:", TEST_PREDICTION_PATH)
print("Test queries:", len(test_predictions))


Encoding TEST candidate pool:   0%|          | 0/432 [00:00<?, ?it/s]

Encoding TEST queries:   0%|          | 0/60 [00:00<?, ?it/s]

Saved: /kaggle/working/pcr_exp3_lora_random/test_predictions_top7.json
Test queries: 237


In [70]:
# ------------------------------------------------------------
# End of PCR Experiment 3
# ------------------------------------------------------------

print("PCR Experiment 3 complete.")
print("Adaptation:", "LoRA + random negatives")
print("random negatives: enabled")
print("MAX_CHUNKS:", MAX_CHUNKS)
print("CHUNK_BATCH_SIZE:", CHUNK_BATCH_SIZE)
print("NUM_NEGATIVES:", NUM_NEGATIVES)


PCR Experiment 3 complete.
Adaptation: LoRA + random negatives
random negatives: enabled
MAX_CHUNKS: 8
CHUNK_BATCH_SIZE: 8
NUM_NEGATIVES: 4
